In [1]:
import re
import random
import sys
from collections import defaultdict
from google.colab import files

uploaded = files.upload()
filepath = list(uploaded.keys())[0]

Saving foxnewshealth.txt to foxnewshealth.txt
Saving usnewshealth.txt to usnewshealth.txt
Saving msnhealthnews.txt to msnhealthnews.txt
Saving bbchealth.txt to bbchealth.txt
Saving everydayhealth.txt to everydayhealth.txt
Saving gdnhealthcare.txt to gdnhealthcare.txt
Saving KaiserHealthNews.txt to KaiserHealthNews.txt
Saving NBChealth.txt to NBChealth.txt
Saving wsjhealth.txt to wsjhealth.txt
Saving nprhealth.txt to nprhealth.txt
Saving latimeshealth.txt to latimeshealth.txt
Saving reuters_health.txt to reuters_health.txt
Saving cnnhealth.txt to cnnhealth.txt
Saving cbchealth.txt to cbchealth.txt
Saving nytimeshealth.txt to nytimeshealth.txt
Saving goodhealth.txt to goodhealth.txt


In [2]:
# Preprocessing

def preprocess_tweet(line: str) -> list[str]:
  parts = line.strip().split("|")
  if len(parts) < 3:
    return []

  text = parts[2]

  # Removing URL's
  text = re.sub(r"https\S+|www\.\S+", "", text)

  # Removing @mentions
  text = re.sub(r"@\w+", "", text)

  # Removing #'s before words
  text = text.replace("#", "")

  # Lowercasing and tokenizing
  words = text.lower().split()

  # Removing Empty strings and puncutation only tokens

  words = [w for w in words if re.search(r"[a-z0-9]", w)]

  return words

def load_tweets(filepath: str) -> list[set]:

  tweets = []
  with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
      words = preprocess_tweet(line)
      if words:
        tweets.append(set(words))
  return tweets


In [3]:
test_line = "575222521002496000|Tue Mar 10 14:07:04|RT @someUser: Check out #health http://t.co/xyz"

output = preprocess_tweet(test_line)

print(output)

['rt', 'check', 'out', 'health', 'http://t.co/xyz']


In [4]:
# Jaccard distance

def jaccard_distance(a: set, b: set) -> float:
  if not a and not b:
    return 1.0
    # word(s) in both tweets
  intersection = len(a & b)
  # Unique words in both tweets
  union = len(a|b)
  return 1.0 - intersection / union


In [5]:
# Centroid Selection

def find_centroid(cluster_tweets: list[set]) -> set:
  if len(cluster_tweets) == 1:
    return cluster_tweets[0]

  best_tweet = None
  best_total = float("inf")

  for candidate in cluster_tweets:
    total = sum(jaccard_distance(candidate, other)
                for other in cluster_tweets)
    if total < best_total:
      best_total = total
      best_tweet = candidate

  return best_tweet

In [6]:
# K-Means Algorithm

def kmeans(tweets: list[set], k: int, max_iter: int = 100,
           seed: int = 42) -> tuple[list[int], list[set], float]:
    random.seed(seed)
    n = len(tweets)

    centroid_indices = random.sample(range(n), k)
    centroids = [tweets[i] for i in centroid_indices]

    labels = [-1] * n

    for iteration in range(max_iter):
      new_labels =[]
      for tweet in tweets:
        distances = [jaccard_distance(tweet, centroid) for centroid in centroids]
        new_labels.append(distances.index(min(distances)))

      if new_labels == labels:
        print(f" Converged at iteration {iteration + 1}")
        break
      labels = new_labels

      clusters: dict[int, list[set]] = defaultdict(list)
      for i, label in enumerate(labels):
        clusters[label].append(tweets[i])

      new_centroids = []
      for i in range (k):
        if clusters[i]:
          new_centroids.append(find_centroid(clusters[i]))
        else:
          new_centroids.append(tweets[random.randint(0, n - 1)])
      centroids = new_centroids

    sse = sum(jaccard_distance(tweets[i], centroids[labels[i]]) ** 2
            for i in range(n)
    )

    return labels, centroids, sse


In [7]:
def report(k: int, labels: list[int], sse: float) -> None:
  cluster_sizes: dict[int, int] = defaultdict(int)
  for i in labels:
    cluster_sizes[i] += 1

  print(f"\n{'='*50}")
  print(f" K = {k}")
  print(f" SSE = {sse:.4f}")
  print(f"Cluster sizes:")
  for i in sorted(cluster_sizes):
    print(f"  Cluster {i + 1:>3}: {cluster_sizes[i]} tweets")


In [8]:
def main(filepath):
  print(f"Loading tweets from: {filepath}")
  tweets = load_tweets(filepath)
  print(f"Total tweets loaded after processing: {len(tweets)}")

  k_vals = [5,10,15,20,25]

  results = []
  for k in k_vals:
    print(f"\nRunning K-Means with K={k}")
    labels, centroids, sse = kmeans(tweets, k=k, max_iter=100, seed=42)
    report(k, labels, sse)
    results.append((k, sse))
  print("\n\n" + "=" * 50)
  print(f"{'K':>5} | {'SSE':>12}")
  print("-" * 50)

  for k, sse in results:
    print(f"{k:>5} | {sse:>12.4f}")
  print("=" * 50)

In [9]:
if __name__ == "__main__":
  main(filepath)

Loading tweets from: foxnewshealth.txt
Total tweets loaded after processing: 2000

Running K-Means with K=5
 Converged at iteration 3

 K = 5
 SSE = 1750.1680
Cluster sizes:
  Cluster   1: 892 tweets
  Cluster   2: 210 tweets
  Cluster   3: 308 tweets
  Cluster   4: 508 tweets
  Cluster   5: 82 tweets

Running K-Means with K=10
 Converged at iteration 4

 K = 10
 SSE = 1709.4772
Cluster sizes:
  Cluster   1: 429 tweets
  Cluster   2: 141 tweets
  Cluster   3: 249 tweets
  Cluster   4: 215 tweets
  Cluster   5: 275 tweets
  Cluster   6: 55 tweets
  Cluster   7: 33 tweets
  Cluster   8: 220 tweets
  Cluster   9: 102 tweets
  Cluster  10: 281 tweets

Running K-Means with K=15
 Converged at iteration 3

 K = 15
 SSE = 1697.9087
Cluster sizes:
  Cluster   1: 293 tweets
  Cluster   2: 128 tweets
  Cluster   3: 303 tweets
  Cluster   4: 99 tweets
  Cluster   5: 95 tweets
  Cluster   6: 56 tweets
  Cluster   7: 33 tweets
  Cluster   8: 192 tweets
  Cluster   9: 68 tweets
  Cluster  10: 318 twe